In [24]:
import re
import os
import nltk
from nltk.corpus import stopwords

import numpy as np
import pandas as pd

from keras.models import Model
from keras.utils import pad_sequences
from keras.preprocessing.text import Tokenizer
from keras.layers import Embedding, LSTM, Input, Dense, Dropout

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [2]:
Max_sequence_length = 50
Embedding_dim = 100

In [3]:
df = pd.read_csv('Emotion_Data.csv')

In [4]:
data = df.sample(n=20000)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 20000 entries, 18093 to 16690
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Text     20000 non-null  object
 1   Emotion  20000 non-null  object
dtypes: object(2)
memory usage: 468.8+ KB


In [6]:
ps = nltk.PorterStemmer()
wl = nltk.WordNetLemmatizer()

In [7]:
def preprocess(text):
    stop = stopwords.words('english')
    text = text.lower()
    text = re.sub('[^a-zA-Z\s]','',text)
    text = [word for word in text.split() if word not in stop]
    text = [ps.stem(wl.lemmatize(word)) for word in text]
    return text

In [8]:
processed_data = data['Text'].map(preprocess)

In [9]:
processed_data.head()

18093                        [feel, like, he, littl, piss]
18095    [feel, like, issu, focu, exposur, late, sure, ...
19639    [feel, embarrass, even, type, absurd, word, tr...
15023                                       [feel, devast]
7925     [know, art, anim, lame, feel, particularli, vi...
Name: Text, dtype: object

In [10]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(processed_data)
sequences = tokenizer.texts_to_sequences(processed_data)

word_index = tokenizer.word_index
len(word_index)

12745

In [11]:
max([len(i) for i in sequences])

35

In [12]:
features = pad_sequences(sequences, Max_sequence_length)
labels = pd.get_dummies(data['Emotion'])

features.shape, labels.shape

((20000, 50), (20000, 6))

In [13]:
labels.head()

,anger,fear,happy,love,sadness,surprise
18093,1,0,0,0,0,0
18095,0,0,1,0,0,0
19639,0,0,0,0,1,0
15023,0,0,0,0,1,0
7925,1,0,0,0,0,0


In [14]:
x_train, x_test, y_train, y_test = train_test_split(features,labels,test_size=0.2,random_state = 5)
x_test, x_val, y_test, y_val = train_test_split(features,labels, test_size=0.5, random_state=3)

In [15]:
embedding_index = {}
with open(r"D:\Notebook\Projects\Embedding_Models\glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coef = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coef

embedding_matrix = np.zeros((len(word_index)+1,Embedding_dim), dtype=int)
for word,i in word_index.items():
    vector = embedding_index.get(word)
    if vector is not None:
        embedding_matrix[i] = vector
        
embedding_layer = Embedding(len(word_index)+1,Embedding_dim, weights = [embedding_matrix],
                            input_length=Max_sequence_length)(input_sequences)

In [19]:
convs = []
filter_sizes = [3,4,5]

sequence_input = Input(shape=(MAX_SEQUENCE_LENGTH,))
embedded_sequences = embedding_layer(sequence_input)

for fsz in filter_sizes:
    l_conv = Conv1D(128,fsz,activation='relu')(embedded_sequences)
    l_pool = MaxPooling1D(5)(l_conv)
    convs.append(l_pool)   
l_merge = Concatenate()(convs)
l_cov1= Conv1D(filters=128, kernel_size=5, activation='relu')(l_merge)
l_pool1 = MaxPooling1D(5)(l_cov1)
# l_cov2 = Conv1D(filters=128, kernel_size=5, activation='relu')(l_pool1)
# l_pool2 = MaxPooling1D(30)(l_cov2)
l_flat = Flatten()(l_pool1)
l_dense = Dense(128, activation='relu')(l_flat)
preds = Dense(13, activation='softmax')(l_dense)

model = Model(sequence_input, preds)
model.compile(loss='binary_crossentropy',
              optimizer='Nadam',
              metrics=['acc'])

model.summary()


In [26]:
history = model.fit(x_train,y_train, validation_data=(x_val,y_val), epochs=25, batch_size=150, verbose=1)

Epoch 1/25
107/107 [==============================] - 78s 726ms/step - loss: 0.3650 - acc: 0.8741 - val_loss: 0.2265 - val_acc: 0.9202
Epoch 2/25
107/107 [==============================] - 76s 713ms/step - loss: 0.1847 - acc: 0.9332 - val_loss: 0.1599 - val_acc: 0.9435
Epoch 3/25
107/107 [==============================] - 78s 731ms/step - loss: 0.1254 - acc: 0.9521 - val_loss: 0.1471 - val_acc: 0.9501
Epoch 4/25
107/107 [==============================] - 75s 698ms/step - loss: 0.0971 - acc: 0.9625 - val_loss: 0.1332 - val_acc: 0.9545
Epoch 5/25
107/107 [==============================] - 75s 706ms/step - loss: 0.0766 - acc: 0.9703 - val_loss: 0.1321 - val_acc: 0.9588
Epoch 6/25
107/107 [==============================] - 73s 680ms/step - loss: 0.0566 - acc: 0.9781 - val_loss: 0.1196 - val_acc: 0.9628
Epoch 7/25
107/107 [==============================] - 73s 687ms/step - loss: 0.0510 - acc: 0.9804 - val_loss: 0.1197 - val_acc: 0.9642
Epoch 8/25
107/107 [==============================] - 7

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 
# list all data in history
print(history.history.keys())
# summarize history for accuracy
plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [27]:
y_pred = model.predict(x_test, verbose=1)

313/313 [==============================] - 34s 96ms/step


In [29]:
def fx(array):
    return np.argmax(array)

temp = np.array(y_test)
st_pred = y_pred.tolist()
st_temp = temp.tolist()

arr = np.array(list(map(fx,st_temp)))
ans = np.array(list(map(fx,st_pred)))

In [30]:
f1 = f1_score(arr,ans, average=None)
f2 = f1_score(arr,ans, average='macro')
f3 =  f1_score(arr,ans, average='micro')
f4 = f1_score(arr,ans, average='weighted')

In [31]:
print(f1)
print(f2)
print(f3)
print(f4)

[0.97379913 0.96623794 0.97546854 0.93422819 0.98246211 0.9522673 ]
0.9640772017124856
0.9721
0.9720206495905821


In [32]:
acc = accuracy_score(arr,ans)
acc

0.9721

In [33]:
p1 = precision_score(arr,ans, average='weighted')
r1 = recall_score(arr,ans,average='weighted')
p1,r1

(0.9720551282777782, 0.9721)

In [34]:
p1 = precision_score(arr,ans, average='macro')
r1 = recall_score(arr,ans,average='macro')
p1,r1

(0.9674064653376702, 0.9609822864769598)

In [35]:
p1 = precision_score(arr,ans, average='micro')
r1 = recall_score(arr,ans,average='micro')
p1,r1

(0.9721, 0.9721)

In [36]:
p1 = precision_score(arr,ans, average=None)
r1 = recall_score(arr,ans,average=None)
p1,r1

(array([0.96956522, 0.9623699 , 0.97206195, 0.95867769, 0.98262943,
        0.95913462]),
 array([0.97807018, 0.97013721, 0.97889908, 0.91099476, 0.98229486,
        0.94549763]))